In [66]:
import pandas as pd
import numpy as np
import re
import ast


In [60]:
HL_base = pd.read_csv("C:/Users/Jesse/OneDrive/Desktop/QSS20_WI26/Final Project/trafficking_states_master.csv")
HL_base = HL_base.drop(columns=["% of total signals", "% of total cases"])
HL_base.head()

,State,Signals received,Cases identified,Year
0,Alabama,231,105,2024
1,Alaska,46,22,2024
2,Arizona,550,300,2024
3,Arkansas,170,80,2024
4,California,3378,1733,2024


In [55]:
ICE_base = pd.read_csv("C:/Users/Jesse/OneDrive/Desktop/QSS20_WI26/Final Project/ICERemovalsdata.csv",
                      encoding='latin-1')


In [56]:
ICE_base = ICE_base.drop(columns=["Criminality", "Arresting Agency", "Criminality", "Country of Citizenship", "Fiscal Quarter", "Fiscal Month", "Month-Year"])

In [57]:
ICE_byyear = ICE_base.groupby(["Area of Responsibility (AOR)", "Fiscal Year"])["Removals"].sum().reset_index()

In [87]:
ICE_byyear

,Area of Responsibility (AOR),Fiscal Year,Removals
0,Atlanta,2021,2780
1,Atlanta,2022,2598
2,Atlanta,2023,5002
3,Atlanta,2024,6277
4,Atlanta,2025,978
...,...,...,...
119,Washington,2021,319
120,Washington,2022,75
121,Washington,2023,353
122,Washington,2024,820


In [25]:
AOR_merge = pd.read_csv("C:/Users/Jesse/OneDrive/Desktop/QSS20_WI26/Final Project/ICEEROmanualdatacleancompleted.csv")

In [61]:
AOR_merge["office_name"] = AOR_merge["office_name"].str.replace(" Field Office", "", regex=False).str.strip()

AOR_merge.head()

,office_name,office_name_short,area
0,Atlanta,Atlanta - ERO,"['Georgia', 'North Carolina', 'South Carolina']"
1,Baltimore,Baltimore - ERO,['Maryland']
2,Buffalo,Buffalo - ERO,['New York']
3,Boston,Burlington - ERO,"['Connecticut', 'Maine', 'Massachusetts', 'New..."
4,Denver,Centennial - ERO,"['Colorado', 'Wyoming']"


In [64]:
AOR_areas_exploded = AOR_merge.explode("area").reset_index(drop=True)

In [69]:
AOR_merge["area"] = AOR_merge["area"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
areas_exploded = AOR_merge.explode("area").reset_index(drop=True)

In [71]:
HL_merged = pd.merge(HL_base, areas_exploded, left_on="State", right_on="area", how="left")

In [77]:
HL_byAOR = HL_merged.groupby(["office_name", "Year"])[["Signals received", "Cases identified"]].sum().reset_index()

In [89]:
HL_ICE_merged = pd.merge(HL_byAOR, ICE_byyear, left_on=["office_name", "Year"], right_on=["Area of Responsibility (AOR)", "Fiscal Year"], how="right")

In [90]:
HL_ICE_merged

,office_name,Year,Signals received,Cases identified,Area of Responsibility (AOR),Fiscal Year,Removals
0,Atlanta,2021.0,2447.0,629.0,Atlanta,2021,2780
1,Atlanta,2022.0,1946.0,573.0,Atlanta,2022,2598
2,Atlanta,2023.0,1467.0,567.0,Atlanta,2023,5002
3,Atlanta,2024.0,1781.0,779.0,Atlanta,2024,6277
4,NaN,NaN,NaN,NaN,Atlanta,2025,978
...,...,...,...,...,...,...,...
119,Washington,2021.0,778.0,185.0,Washington,2021,319
120,Washington,2022.0,690.0,199.0,Washington,2022,75
121,Washington,2023.0,685.0,204.0,Washington,2023,353
122,Washington,2024.0,796.0,283.0,Washington,2024,820
